# Inspect Aggregated Snow-Covered Area

HRU-level inspection of MOD10C1 snow-covered area. Mirrors `inspect_consolidated_snow_covered_area.ipynb`.

Source (see `catalog/variables.yml` → `snow_covered_area`):

- MOD10C1 v061: `Day_CMG_Snow_Cover` (SCA, 0–100% with flag values), `Day_CMG_Clear_Index` (CI, 0–100% with flag values). Also carries `Day_CMG_Cloud_Obscured` (QA cross-check) and `Snow_Spatial_QA` (categorical 0–4, **not** the CI).

See `docs/references/calibration-target-recipes.md` §5 for flag-value handling and the CI > 0.70 filter.

## Per-target conventions in this notebook

- Mask `Day_CMG_Snow_Cover` and `Day_CMG_Clear_Index` flag values: 107 (lake ice), 111 (night), 237 (inland water), 239 (ocean), 250 (cloud-obscured water), 253 (data not mapped), 255 (fill). Without the mask, CONUS-mean SCA on a typical day is dominated by 237/239/250 codes and lands near 100, which is meaningless.
- `sca = Day_CMG_Snow_Cover / 100`, `ci = Day_CMG_Clear_Index / 100`.
- `Snow_Spatial_QA` is categorical 0–4 plus flag codes — **NOT a percentage CI**. Earlier catalog versions had `ci = Snow_Spatial_QA / 100; ci > 0.70` which would pass *only* the special-case flag codes. We do not use `Snow_Spatial_QA` for any quantitative purpose.
- Cell 7 shows **two maps side by side**: the raw daily SCA at `TARGET_DAY` (with flag values masked, no CI filter) and the **monthly mean composite** of valid (CI > 0.70) `Day_CMG_Snow_Cover`. The CI-filter contrast is the most useful diagnostic in this notebook.
- Cells 9–13 use the CI-filtered monthly composite.
- If MOD10C1 has not been re-aggregated post-PR-#68 to include `Day_CMG_Clear_Index`, the notebook degrades gracefully: the raw daily map runs, but the CI-filtered composite is skipped with a clear message.
- This is a single-source target — no cross-source colour scale; we use a fixed `vmin=0, vmax=1`.

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
# import _helpers
# _helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "snow_covered_area"
TARGET_DAY = "2000-01-15"
TARGET_YEAR = 2000
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2006)  # daily data — 6 years is enough
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}
SOURCES = {
    "mod10c1_v061": {
        "label": "MOD10C1 v061",
        "var": "Day_CMG_Snow_Cover",
        "ci_var": "Day_CMG_Clear_Index",
    },
}

MOD10C1_FLAGS = {107, 111, 237, 239, 250, 253, 255}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get('nhm_id', ds.sizes.get('hru_id', 'unknown'))} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

In [ ]:
def _mask_flags(da: xr.DataArray) -> xr.DataArray:
    """Mask MOD10C1 flag values (>100) to NaN."""
    return da.where(~da.isin(list(MOD10C1_FLAGS)))


def _ci_filtered_monthly_mean(ds: xr.Dataset, year: int, month: int):
    """Compute monthly mean of valid (CI > 0.70) Day_CMG_Snow_Cover.

    Returns None if Day_CMG_Clear_Index is not in the aggregated NC
    (e.g. MOD10C1 hasn't been re-aggregated post-PR-#68).
    """
    if "Day_CMG_Clear_Index" not in ds.data_vars:
        return None
    sca = _mask_flags(ds["Day_CMG_Snow_Cover"]) / 100.0
    ci = _mask_flags(ds["Day_CMG_Clear_Index"]) / 100.0
    sca_valid = sca.where(ci > 0.70)
    sca_month = sca_valid.sel(time=slice(
        pd.Timestamp(year=year, month=month, day=1),
        pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    ))
    return sca_month.mean("time", skipna=True).to_pandas()


if not opened:
    print("No sources available; skipping native-unit maps.")
else:
    ds, info = opened["mod10c1_v061"]
    da_day = _mask_flags(ds[info["var"]].sel(time=TARGET_DAY, method="nearest")) / 100.0
    monthly = _ci_filtered_monthly_mean(ds, TARGET_YEAR, TARGET_MONTH)

    if monthly is None:
        fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
        plot_hru_choropleth(
            axes[0, 0], fabric, da_day.to_pandas(),
            vmin=0, vmax=1, cmap="Blues",
            title=f"Day_CMG_Snow_Cover (raw, flags masked)\n{TARGET_DAY}",
            units="fraction",
        )
        print("SKIP CI-filtered composite: Day_CMG_Clear_Index not in aggregated NC. "
              "Re-aggregate MOD10C1 to pick up post-PR-#68 catalog changes.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), squeeze=False)
        plot_hru_choropleth(
            axes[0, 0], fabric, da_day.to_pandas(),
            vmin=0, vmax=1, cmap="Blues",
            title=f"Day_CMG_Snow_Cover (raw, flags masked)\n{TARGET_DAY}",
            units="fraction",
        )
        plot_hru_choropleth(
            axes[0, 1], fabric, monthly,
            vmin=0, vmax=1, cmap="Blues",
            title=f"CI-filtered monthly mean (CI > 0.70)\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
            units="fraction",
        )
    fig.suptitle("Snow-covered area — daily raw vs. monthly CI-filtered composite", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

## Monthly CI-filtered composite map

The CI-filtered composite (`Day_CMG_Clear_Index > 0.70`) is what the SCA target builder ultimately consumes. We plot it here as a single panel on a fixed [0, 1] colour scale; this is also the basis for the cross-source-distribution histogram (cell 10) and the validation cell (cell 13).

If `Day_CMG_Clear_Index` is missing from the aggregated NC, this and downstream cells skip with a clear message.

In [ ]:
if opened and monthly is not None:
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, monthly,
        vmin=0, vmax=1, cmap="Blues",
        title=f"MOD10C1 v061 — CI-filtered monthly SCA\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="fraction",
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()
elif opened:
    print("CI-filtered composite unavailable; see cell 7.")

In [ ]:
if opened and monthly is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(monthly.dropna(), bins=60, histtype="step", linewidth=2, density=True)
    ax.set_xlabel("Snow-covered area (fraction)")
    ax.set_ylabel("Density")
    ax.set_title(f"HRU-level SCA distribution — {TARGET_YEAR}-{TARGET_MONTH:02d} (CI > 0.70)")
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    ds_range = open_year_range(project_dir, "mod10c1_v061", TIME_SERIES_YEARS)
    try:
        id_dim = "nhm_id" if "nhm_id" in ds_range.dims else "hru_id"
        sca_full = ds_range["Day_CMG_Snow_Cover"].sel({id_dim: list(rep_hrus.values())}).load()
        ci_full = (
            ds_range["Day_CMG_Clear_Index"].sel({id_dim: list(rep_hrus.values())}).load()
            if "Day_CMG_Clear_Index" in ds_range.data_vars else None
        )
    finally:
        ds_range.close()

    sca_full = _mask_flags(sca_full) / 100.0
    if ci_full is not None:
        ci_full = _mask_flags(ci_full) / 100.0
        sca_full = sca_full.where(ci_full > 0.70)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        ax.plot(sca_full.time, sca_full.sel({id_dim: hru_id}).values, linewidth=0.8)
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("SCA fraction")
        ax.set_ylim(0, 1)
    fig.suptitle(
        f"SCA at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)} "
        f"(CI > 0.70 mask {'on' if ci_full is not None else 'OFF — recipe re-aggregation needed'})"
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

In [ ]:
if opened:
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    coverage_basis = monthly if monthly is not None else da_day.to_pandas()
    n_nan = nan_hru_count(coverage_basis)
    print(f"MOD10C1 v061: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
    plot_nan_hrus(
        axes[0, 0],
        fabric,
        coverage_basis,
        title=f"NaN HRUs (red) — {n_nan} of {len(fabric)} "
              f"({'CI-filtered monthly' if monthly is not None else 'raw daily'})",
    )
    fig.suptitle("Coverage gaps — NN-filled in normalize/", fontsize=12, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

In [ ]:
def _gridded_mean_sca(year, month):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox; HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal.
    """
    path = datastore_dir / "mod10c1_v061" / f"mod10c1_v061_{year}.nc"
    if not path.exists():
        return None, f"missing consolidated NC at {path}"
    with xr.open_dataset(path) as ds:
        if "Day_CMG_Clear_Index" not in ds.data_vars:
            return None, "Day_CMG_Clear_Index missing in consolidated NC"
        sca = _mask_flags(ds["Day_CMG_Snow_Cover"]) / 100.0
        ci = _mask_flags(ds["Day_CMG_Clear_Index"]) / 100.0
        sca_valid = sca.where(ci > 0.70).sel(time=slice(
            pd.Timestamp(year=year, month=month, day=1),
            pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
        )).load()
    return float(sca_valid.mean(skipna=True).item()), None


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
if opened and monthly is not None:
    agg_mean = area_weighted_mean(monthly, fabric)
    gridded_mean, reason = _gridded_mean_sca(TARGET_YEAR, TARGET_MONTH)
    if gridded_mean is None:
        print(f"{'MOD10C1 v061':<35} {agg_mean:>12.4f}  {'SKIP':>12} ({reason})")
    else:
        diff = agg_mean - gridded_mean
        pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
        print(f"{'MOD10C1 v061':<35} {agg_mean:>12.4f} {gridded_mean:>12.4f} {diff:>12.4f} {pct:>7.2f}%")
else:
    print("Validation skipped — CI-filtered composite unavailable.")

## Why HRU-level patterns matter

MOD10C1 is the only SCA source for the calibration target, so this notebook isn't comparing across sources — it's a coverage / quality diagnostic.

- **The flag-value mask is critical.** Cell 7's raw daily map (left) shows what happens *with* the mask but *without* the CI filter — open water, cloud cover, and night codes are NaN, but legitimate cloud-obscured-snow days are still in the mean. The CI-filtered monthly composite (right) drops those days; the difference is most visible over the Great Lakes and the Pacific NW.
- **Without the CI filter, a winter monthly mean** can be dominated by `Day_CMG_Snow_Cover = 100` from cloud-obscured-water flag codes that survive a too-narrow mask.
- **`Snow_Spatial_QA` is categorical, not a percent CI.** Earlier catalog versions used it as a CI filter; this passes only the flag codes and rejects every legitimate observation. The notebook never uses `Snow_Spatial_QA` quantitatively.

**Calibration target implication.** TM 6-B10 derives upper/lower bounds from the daily SCA value and the associated CI ("error bound based on the daily snow-covered area value and the associated confidence interval"). The exact formula is unconfirmed (PRMSobjfun.f is not publicly available — open methodology gap). What this notebook checks is that the input to that formula — daily masked SCA with CI > 0.70 — is sane on the HRU fabric.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()